# Лекція 12 — NumPy та векторизація на реальних даних

**Курс:** Applied Software Development (Python) 2026 ·

---

## Передумови

Ця лекція **самодостатня** і не залежить від коду проєкту з лекцій 6–10. Нам не потрібні ані вебфреймворк, ані база даних, ані контейнери.

Достатньо знань з **Лекцій 1–5** плюс мінімальної pandas-інтуїції з **Лекції 11**:

- типи даних, змінні, f-strings (Л1)
- колекції, цикли, генератори списків (Л2–Л3)
- функції, `*args`/`**kwargs`, lambda (Л3–Л4)
- базове файлове введення-виведення (Л5)
- `pd.read_csv`, вибір колонок, `.dropna` (Л11) — лише як "чорний ящик" в одній комірці

Плюс встановлений **Jupyter** і можливість запустити `pip install`.

---

## Як побудована ця лекція

Дві частини:

1. **NumPy як інструмент**: чому він швидкий, як працює `ndarray`, що таке broadcasting, навіщо `%timeit`.
2. **Практичні приклади на реальних даних**: описова статистика, виявлення викидів через IQR, агрегації по групах, top-K, функції втрат і pairwise-distances — усе на NumPy, без жодного ML-фреймворка.

> **Машинне навчання тут навмисно відсутнє** — повноцінний ML-курс ви матимете наступного року. Сьогодні фокус — **інструмент NumPy** та як він робить роботу з числовими даними швидкою й читабельною.

> **Датасет:** ми **повторно використовуємо** CSV Stack Overflow Developer Survey 2025, який ви вже завантажили для Лекції 11 (`lectures/11-pandas-analytics/data/survey_results_public.csv`).
>
> Файл є обов'язковим — без нього ноутбук зупиниться з повідомленням про те, де його взяти. Інструкція з завантаження — у `lectures/11-pandas-analytics/README.md`.


In [23]:
import numpy as np
import random as rnd

m1 = [[rnd.randint(1, 10) for _ in range(1000)] for _ in range(1000)]
m2 = [[rnd.randint(1, 10) for _ in range(1000)] for _ in range(1000)]

In [24]:
def dot_matrix(m1, m2):
    if not m1 or not m2 or len(m1[0]) != len(m2):
        raise ValueError("Incompatible matrix dimensions")
    result = []
    for i in range(len(m1)):
        row = []
        for j in range(len(m2[0])):
            sum = 0
            for k in range(len(m2)):
                sum += m1[i][k] * m2[k][j]
            row.append(sum)
        result.append(row)
    return result

def dot_matrices_numpy(m1, m2):
    return np.dot(m1, m2)

In [25]:
%timeit dot_matrix(m1, m2)

KeyboardInterrupt: 

In [27]:
%timeit dot_matrices_numpy(m1, m2)

574 ms ± 14.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Цілі заняття

Після цієї лекції ви зможете:

1. Пояснити, **чому NumPy швидший за Python list** — суцільна пам'ять, фіксовані dtypes, диспатч до C.
2. Створювати та маніпулювати `ndarray` — індексація, зрізи, fancy indexing, boolean masking, broadcasting.
3. Виміряти різницю у швидкості векторизованого коду через `%timeit` і свідомо вирішувати, коли вектор економить час, а коли — ні.
4. **Робити описову статистику** на числових масивах — `mean`, `median`, `np.percentile`, IQR-фільтр викидів.
5. **Агрегувати по групах** через boolean masking, шукати топ-K через `np.argsort`, рахувати частоти через `np.unique(return_counts=True)`.
6. Реалізувати **функції втрат (MSE, MAE, RMSE)** одним рядком NumPy і пояснити, чому це у сотні разів швидше за Python-цикл.
7. Побудувати **матрицю парних відстаней** через broadcasting — без жодного циклу.


## Чому NumPy швидкий?

Якщо коротко — **через три речі**, які Python list зробити не може:

### 1. Суцільна пам'ять (contiguous memory)

Python `list[int]` — це масив **посилань** на окремі int-об'єкти, розкидані по купі. Кожен int — повноцінний PyObject з лічильником посилань, типом і значенням; це 28 байт замість 8.

NumPy `ndarray` — це один **суцільний шматок пам'яті** з фіксованим dtype. Мільйон `int64`-чисел — це рівно 8 МБ підряд, без жодного зайвого байта. Процесор любить таку пам'ять: префетчер передбачає наступні читання, кеш-лінії заповнюються одна за одною.

### 2. Фіксований dtype

У Python list елементи можуть бути будь-якого типу — `[1, "two", 3.14, None]` валідний. Це гнучко, але для математики дорого: щоразу інтерпретатор має спитати "а це що за об'єкт?".

В `ndarray` тип фіксований під час створення — `int32`, `float64`, `bool`. Ніяких перевірок типу на кожному елементі під час циклу.

### 3. Диспатч до C / SIMD

Коли ви пишете `arr ** 2` на Python-списку, інтерпретатор робить ~мільйон викликів `__mul__` — кожен з оверхедом інтерпретатора (десятки наносекунд на ітерацію).

`ndarray ** 2` робить **один** виклик C-функції, яка проганяє SIMD-інструкції процесора по суцільному масиву. SIMD = "Single Instruction, Multiple Data" — одна інструкція обробляє 4 або 8 чисел одночасно.

> **Перевіримо це експериментально** в Розділі 8 — там ми зробимо `%timeit`-бенчмарк і побачимо ~100×–500× різницю на мільйонному масиві.


## Константи лекції та перевірка середовища

Закріпимо всі константи в одному місці на самому початку — щоб усі шляхи й seeds легко знайти й змінити.

In [32]:
from pathlib import Path

import numpy as np
import pandas as pd

# --- Шляхи ---
# Шукаємо CSV у L11 спочатку відносно теки лекції, потім відносно cwd.
_HERE = Path.cwd()
_CSV_CANDIDATES = [
    Path("../11-pandas-analytics/data/survey_results_public.csv"),
    Path("lectures/11-pandas-analytics/data/survey_results_public.csv"),
    _HERE.parent / "11-pandas-analytics" / "data" / "survey_results_public.csv",
]
SURVEY_CSV_PATH = next((p for p in _CSV_CANDIDATES if p.exists()), _CSV_CANDIDATES[0])

# --- Seed для всіх випадкових генераторів цієї лекції ---
SEED = 42

# --- Перевірка середовища ---
print(f"NumPy:  {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Survey CSV path: {SURVEY_CSV_PATH}")
print(f"Survey CSV exists: {SURVEY_CSV_PATH.exists()}")


NumPy:  2.4.4
pandas: 3.0.2
Survey CSV path: ../11-pandas-analytics/data/survey_results_public.csv
Survey CSV exists: False


## Основи `ndarray`: створення та властивості

`ndarray` — це **n-вимірний масив** фіксованого dtype. Є кілька канонічних способів його створити.

In [28]:
# Зі звичайного Python-списку
a = np.array([1, 2, 3, 4, 5])
print(a, a.dtype)


[1 2 3 4 5] int64


In [29]:
# Заповнити нулями / одиницями — корисно для ініціалізації параметрів
zeros = np.zeros((3, 4))      # 2-D масив 3×4
ones = np.ones(5, dtype=np.int32)
print("zeros shape:", zeros.shape, "dtype:", zeros.dtype)
print("ones:", ones)


zeros shape: (3, 4) dtype: float64
ones: [1 1 1 1 1]


In [30]:
# Прогресії
print(np.arange(0, 10, 2))         # від 0 до 10 з кроком 2 (виключно)
print(np.linspace(0, 1, num=5))    # 5 рівномірних точок від 0 до 1 (включно)


[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


In [40]:
# Випадкові числа — сучасний API через Generator
rng = np.random.default_rng(seed=SEED)
sample = rng.standard_normal((2, 3))   # стандартний нормальний розподіл, форма 2×3
print(sample)


[[ 0.30471708 -1.03998411  0.7504512 ]
 [ 0.94056472 -1.95103519 -1.30217951]]


### Атрибути `ndarray`, які ви бачитимете щодня

| Атрибут | Що показує |
|---------|------------|
| `.shape` | кортеж розмірностей: `(3, 4)` для 2-D масиву 3×4 |
| `.dtype` | тип даних: `int64`, `float64`, `bool`, … |
| `.ndim` | кількість вимірів: 1, 2, 3, … |
| `.size` | загальна кількість елементів (= `prod(.shape)`) |
| `.nbytes` | скільки байтів масив займає в пам'яті |


In [41]:
matrix = rng.standard_normal((100, 50))
print(f"shape:  {matrix.shape}")
print(f"dtype:  {matrix.dtype}")
print(f"ndim:   {matrix.ndim}")
print(f"size:   {matrix.size}")
print(f"nbytes: {matrix.nbytes:,} (= {matrix.size} × {matrix.itemsize} байт)")


shape:  (100, 50)
dtype:  float64
ndim:   2
size:   5000
nbytes: 40,000 (= 5000 × 8 байт)


## dtype: коли важливо обрати правильний

Більшість коду працює з `int64` чи `float64` за замовчуванням — і це правильно. Але іноді dtype має значення:

- **Пам'ять:** float32 займає вдвічі менше за float64. На матриці `(10000, 1000)` це різниця у 40 МБ.
- **Точність:** float32 має ≈7 значущих цифр, float64 — ≈15. Для фінансових розрахунків float32 неприйнятний.
- **Сумісність з GPU / ML-фреймворками:** PyTorch і TensorFlow часто очікують float32 за замовчуванням.

Перетворення — через `.astype()`.

In [42]:
big = np.arange(1_000_000, dtype=np.int64)
big32 = big.astype(np.int32)
print(f"int64: {big.nbytes / 1e6:.2f} MB")
print(f"int32: {big32.nbytes / 1e6:.2f} MB  (вдвічі менше)")


int64: 8.00 MB
int32: 4.00 MB  (вдвічі менше)


## Індексація та зрізи: view vs copy

NumPy має **три** способи "вибрати елементи":

1. **Basic slicing** (`arr[1:4]`, `arr[:, ::2]`) — повертає **view** на ту саму пам'ять.
2. **Fancy indexing** (`arr[[0, 2, 5]]`) — повертає **copy**.
3. **Boolean masking** (`arr[arr > 0]`) — повертає **copy**.

Різниця критична: якщо ви модифікуєте view, оригінал змінюється. Якщо модифікуєте copy — ні.

In [43]:
arr2d = np.arange(20).reshape(4, 5)
print(arr2d)


[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]


In [44]:
# Basic slicing — 1-D і 2-D
print("rows 1-3, every 2nd col:")
print(arr2d[1:3, ::2])


rows 1-3, every 2nd col:
[[ 5  7  9]
 [10 12 14]]


In [45]:
# Fancy indexing — обираємо рядки за списком індексів
print("rows 0, 2, 3:")
print(arr2d[[0, 2, 3]])


rows 0, 2, 3:
[[ 0  1  2  3  4]
 [10 11 12 13 14]
 [15 16 17 18 19]]


In [46]:
# Boolean masking — обираємо елементи за умовою
big_values = arr2d[arr2d > 12]
print("елементи більші за 12:", big_values)


елементи більші за 12: [13 14 15 16 17 18 19]


### View vs copy — наочно

Це найпоширеніша "тиха" помилка з NumPy. Дивіться:

In [48]:
original = np.arange(10)
view = original[2:6]       # basic slicing → view
view[0] = -999             # модифікуємо view
print("original:", original)   # ⚠️ original теж змінився
print("view:    ", view)


original: [   0    1 -999    3    4    5    6    7    8    9]
view:     [-999    3    4    5]


In [ ]:
original = np.arange(10)
copy = original[[2, 3, 4, 5]]   # fancy indexing → copy
copy[0] = -999                  # модифікуємо copy
print("original:", original)    # original НЕ змінився
print("copy:    ", copy)


**Запам'ятайте:** basic slicing = view, fancy/boolean = copy. Якщо потрібен незалежний масив після basic slicing — викличте `.copy()` явно.

## Broadcasting: правила та приклади

Broadcasting — це механізм, який дозволяє виконувати операції над масивами **різних форм** без явного копіювання даних.

### Правила (читати справа наліво)

При операції над двома масивами NumPy порівнює їхні форми поелементно з кінця:

1. Якщо розмірності **рівні** — все ок, операція поелементна.
2. Якщо одна з розмірностей дорівнює **1** — вона "розтягується" до іншої.
3. Якщо одна форма **коротша** — спереду додаються одиниці (так само "розтягуються").
4. Інакше — **`ValueError`**.

Простіше показати на прикладах.

In [54]:
# Приклад 1: (3, 4) + (4,) → (3, 4)
# Вектор довжини 4 розтягується вздовж першої осі
M = np.ones((3, 4))
v = np.array([[10, 20, 30, 40]])
print(M + v)


[[11. 21. 31. 41.]
 [11. 21. 31. 41.]
 [11. 21. 31. 41.]]


In [51]:
M

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [55]:
v.shape

(1, 4)

In [56]:
# Приклад 2: (3, 1) + (1, 4) → (3, 4)
# Класичний "outer-add" — стовпець плюс рядок утворюють матрицю
col = np.array([[1], [2], [3]])      # shape (3, 1)
row = np.array([[10, 20, 30, 40]])   # shape (1, 4)
print(col + row)


[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


In [57]:
col

array([[1],
       [2],
       [3]])

In [58]:
# Приклад 3: (5,) + scalar → (5,)
# Скаляр — це фактично shape (), яка broadcast-иться куди завгодно
print(np.array([1, 2, 3, 4, 5]) + 100)


[101 102 103 104 105]


In [59]:
# Приклад 4: НЕ працює — (3, 4) + (3,)
# Праві осі (4 і 3) не рівні і жодна не дорівнює 1
M = np.ones((3, 4))
bad = np.array([1, 2, 3])
try:
    M + bad
except ValueError as e:
    print(f"ValueError: {e}")


ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


**Інтуїція:** broadcasting — це "віртуальне" розтягування без копіювання даних у пам'яті. Це і швидко, і елегантно. Ми будемо його активно використовувати у Розділі 10 (стандартизація: `(X - mean) / std` робить broadcasting `(n, p) - (p,)`).

## Поелементні, редукційні та лінійно-алгебраїчні операції

Це робочий мінімум NumPy-операцій, які ми використовуватимемо до кінця лекції.

In [ ]:
# Поелементні: працюють "пометрово"
x = np.array([1.0, 2.0, 4.0, 16.0])
print("exp:", np.exp(x))   # знадобиться у sigmoid
print("log:", np.log(x))   # знадобиться у BCE loss


### Редукції та осі

Запам'ятайте одне правило: **`axis=k` — це вісь, яка зникне**.

Для 2-D масиву форми `(rows, cols)`:

- `axis=0` колапсує вимір `rows` → результат форми `(cols,)` — це **column sums** (одне число на колонку).
- `axis=1` колапсує вимір `cols` → результат форми `(rows,)` — це **row sums** (одне число на рядок).

Без аргументу `axis=` редукція зведе масив до одного скаляра.

In [ ]:
grid = np.array([[1, 2, 3, 4],
                  [5, 6, 7, 8],
                  [9, 10, 11, 12]])
print("grid.shape =", grid.shape)
print("sum axis=0 → column sums, форма (4,):", grid.sum(axis=0))
print("sum axis=1 → row sums,    форма (3,):", grid.sum(axis=1))
print("mean axis=0 (по колонках):", grid.mean(axis=0))
print("std  axis=1 (по рядках):  ", grid.std(axis=1))
print("argmax (без осі, скаляр): ", grid.argmax())


### Лінійна алгебра: `np.dot` і `@`

Для нашої моделі `ŷ = σ(X·w + b)` потрібен матрично-векторний добуток. У NumPy є два способи його записати:

- `np.dot(a, b)` — старий API
- `a @ b` — оператор Python 3.5+, ідентичний для 1-D і 2-D масивів

Для матриць та векторів **завжди обирайте `@`** — він візуально нагадує математичний запис.

In [ ]:
X = rng.standard_normal((4, 3))     # 4 зразки, 3 ознаки
w = np.array([1.0, -2.0, 0.5])
b = 0.1

# Два способи — той самий результат
print("np.dot(X, w):", np.dot(X, w))
print("X @ w:       ", X @ w)
print("X @ w + b:   ", X @ w + b)   # broadcasting додає скаляр до вектора


> **Дрібний нюанс:** `np.dot` і `@` дають однакові результати для 1-D і 2-D операндів. Розходяться вони лише на тензорах вищого порядку (3-D+) — там `np.dot` має sum-product семантику, а `@` робить batched matmul. У цій лекції ми працюємо тільки з 1-D і 2-D, тож обидва варіанти еквівалентні.

## Швидкість: `%timeit` Python vs NumPy

![Is It Worth the Time? — xkcd 1205](https://imgs.xkcd.com/comics/is_it_worth_the_time.png)

*[xkcd 1205 — Is It Worth the Time?](https://xkcd.com/1205/) · Randall Munroe · [CC BY-NC 2.5](https://xkcd.com/license.html)*

Час побачити, чому NumPy існує. Зробимо одне й те саме завдання — піднести мільйон чисел до квадрату — двома способами і виміряємо.

In [ ]:
data = list(range(1_000_000))

In [ ]:
%%timeit
for x in data:
    x = x ** 2

In [ ]:
arr = np.arange(1_000_000)
%timeit arr ** 2


 NumPy is ~64× faster 

### Чому така велика різниця?

Python-цикл інтерпретується **по одному елементу**: щоразу береться об'єкт `int`, викликається його `__mul__`, повертається новий об'єкт `int`. Це десятки наносекунд оверхеду на кожну ітерацію — мільйон ітерацій = десятки мілісекунд.

NumPy-операція `arr ** 2` робить **один** виклик C-функції, яка проганяє SIMD-інструкції процесора по суцільному масиву. Накладні витрати — стала ~мікросекунда; решта — чиста математика на швидкості пам'яті.

> На дуже маленьких масивах (десятки елементів) NumPy може програти через свій сталий оверхед. Векторизуйте там, де це **природно** — від кількох сотень елементів і більше. Не перетворюйте на `ndarray` все підряд із принципу.

## NumPy на даних Survey 2025

![Statistics — xkcd 2400](https://imgs.xkcd.com/comics/statistics.png)

*[xkcd 2400 — Statistics](https://xkcd.com/2400/) · Randall Munroe · [CC BY-NC 2.5](https://xkcd.com/license.html)*

Час перейти від синтетичних `np.arange` до реального датасету. Ми використаємо `pandas` лише для читання CSV (як у Лекції 11), а далі **усе** робитимемо на чистому NumPy.

### Які колонки беремо

| # | Колонка CSV | Що це |
|---|-------------|-------|
| 1 | `Country` | Країна респондента (рядок — для групування) |
| 2 | `YearsCode` | Скільки років пише код |
| 3 | `WorkExp` | Скільки років професійного досвіду |
| 4 | `ConvertedCompYearly` | Річна компенсація у USD (наша головна числова колонка) |

`pandas` тут — просто завантажувач. Як тільки ми отримаємо `ndarray`, далі чисто NumPy.


In [ ]:
# --- Завантаження Survey CSV ---
USECOLS = ["Country", "YearsCode", "WorkExp", "ConvertedCompYearly"]

if not SURVEY_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Survey CSV не знайдено за шляхом {SURVEY_CSV_PATH}. "
        "Завантажте його за інструкцією з lectures/11-pandas-analytics/README.md "
        "(розділ 'Download the 2025 Stack Overflow Annual Developer Survey')."
    )

df = pd.read_csv(SURVEY_CSV_PATH, usecols=USECOLS)
df = df.dropna(subset=["Country", "ConvertedCompYearly"]).copy()

# pandas → numpy одним викликом .to_numpy()
salaries = df["ConvertedCompYearly"].to_numpy(dtype=np.float64)
years_code = pd.to_numeric(df["YearsCode"], errors="coerce").to_numpy()
work_exp = pd.to_numeric(df["WorkExp"], errors="coerce").to_numpy()
countries = df["Country"].to_numpy()  # масив рядків (object dtype)

print(f"Завантажено {len(salaries):,} рядків зі Survey")
print(f"salaries.dtype  = {salaries.dtype},  shape = {salaries.shape}")
print(f"countries.dtype = {countries.dtype}, унікальних: {len(np.unique(countries))}")


## Швидка статистика: `mean`, `median`, `np.percentile`

Перш ніж щось робити з даними — корисно подивитись на їхній розподіл. NumPy дає це **одним рядком**.

- `arr.mean()` — середнє арифметичне; чутливе до викидів.
- `np.median(arr)` — 50-та персентиль; стійке до викидів.
- `np.percentile(arr, q)` — будь-яка персентиль (`q` — від 0 до 100, можна списком).
- `arr.std(ddof=0)` — стандартне відхилення (популяційне; `ddof=1` для вибіркового).

> **Важливо для зарплат:** mean і median різко розходяться, бо розподіл сильно скошений вправо (кілька топ-зарплат тягнуть середнє вгору). Це класичний сигнал, що саме median — більш репрезентативна "типова" зарплата.


In [ ]:
# Описова статистика salaries — усе одним рядком NumPy
print(f"Кількість записів: {len(salaries):,}")
print(f"Mean:    ${salaries.mean():>12,.0f}")
print(f"Median:  ${np.median(salaries):>12,.0f}")
print(f"Std:     ${salaries.std(ddof=0):>12,.0f}")
print(f"Min:     ${salaries.min():>12,.0f}")
print(f"Max:     ${salaries.max():>12,.0f}")

# Кілька персентилів одним викликом — q приймає масив
percentiles = np.percentile(salaries, [10, 25, 50, 75, 90, 99])
print("\nПерсентилі:")
for q, val in zip([10, 25, 50, 75, 90, 99], percentiles):
    print(f"  p{q:>2}: ${val:>12,.0f}")


**Що ми бачимо:** mean набагато більший за median — типовий ознака правоскошеного розподілу з товстим хвостом великих зарплат. p99 у рази більший за p90 — хвіст реально товстий. Тепер логічне наступне питання: **скільки там викидів і де провести межу?**

## Виявлення викидів через IQR

![Box Plot — xkcd 1798](https://imgs.xkcd.com/comics/box_plot.png)

*[xkcd 1798 — Box Plot](https://xkcd.com/1798/) · Randall Munroe · [CC BY-NC 2.5](https://xkcd.com/license.html)*

**IQR (Inter-Quartile Range)** — відстань між 25-ю та 75-ю персентилями. Класичний фільтр Тукі: усе, що далі за `1.5 · IQR` від найближчого квартиля — кандидати на викиди.

$$
\text{IQR} = Q_3 - Q_1
\qquad
\text{lower} = Q_1 - 1.5 \cdot \text{IQR}
\qquad
\text{upper} = Q_3 + 1.5 \cdot \text{IQR}
$$

Чому саме 1.5 — емпіричне правило Джона Тукі (для нормального розподілу це ≈ 2.7σ, флагує ~0.7% хвостів). Для скошених розподілів (як зарплати) часто беруть 3.0.


In [ ]:
# IQR-фільтр викидів — три рядки на повністю векторизованому коді
q1, q3 = np.percentile(salaries, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Boolean masking — серце NumPy: умова → масив True/False → лічимо/фільтруємо
outlier_mask = (salaries < lower) | (salaries > upper)
n_outliers = int(outlier_mask.sum())            # True == 1, False == 0
share = outlier_mask.mean() * 100                # частка True у відсотках

print(f"Q1 = ${q1:,.0f},  Q3 = ${q3:,.0f},  IQR = ${iqr:,.0f}")
print(f"Межі (1.5·IQR): [${lower:,.0f}, ${upper:,.0f}]")
print(f"Викидів: {n_outliers:,} з {len(salaries):,} ({share:.2f}%)")

# Та сама ідея — але "чисті" зарплати без викидів
clean = salaries[~outlier_mask]
print(f"\nMedian усіх:           ${np.median(salaries):,.0f}")
print(f"Median без викидів:    ${np.median(clean):,.0f}")
print(f"Mean усіх:             ${salaries.mean():,.0f}")
print(f"Mean без викидів:      ${clean.mean():,.0f}  ← змінився сильніше: mean чутливий до хвостів")


## Per-country середня зарплата: цикл vs векторизація

Класична задача аналітики: **порахувати середнє по групах**. Зробимо її двома способами на справжніх даних Survey і виміряємо різницю.

**Постановка.** Дано `salaries` (зарплати) та `countries` (країна кожного респондента). Порахувати середню зарплату для кожної країни.

**Підхід 1 — наївний Python-цикл** по кожному респонденту: словник `{країна: [сума, кількість]}`.
**Підхід 2 — векторизація**: для кожної унікальної країни — `salaries[codes == c].mean()`.

> **Pro tip.** Перед `==`-порівнянням ми **закодуємо** назви країн у цілі числа через `np.unique(..., return_inverse=True)`. Без цього NumPy порівнює object-dtype рядки **поелементно у Python** — і вектор працює повільніше за наївний цикл. З int-кодами `==` запускається у C — і швидкість зростає на порядок.


In [ ]:
# Беремо лише топ-10 найчисленніших країн — щоб результат був читабельним
top_countries = (
    pd.Series(countries).value_counts().head(10).index.to_numpy()
)
mask_top = np.isin(countries, top_countries)
sal_top = salaries[mask_top]
ctr_top = countries[mask_top]

# Кодуємо назви країн у цілі коди ОДИН раз. np.unique(..., return_inverse=True)
# повертає (унікальні_назви, індекси_у_цьому_масиві_для_кожного_елемента).
country_names, country_codes = np.unique(ctr_top, return_inverse=True)
print(f"Працюємо з {len(sal_top):,} рядками з {len(country_names)} країн")
print(f"country_codes.dtype = {country_codes.dtype} ← int, не object!")


In [ ]:
# (a) Чесний наївний Python-цикл — по КОЖНОМУ респонденту
def loop_means(sals, codes):
    sums: dict[int, float] = {}
    counts: dict[int, int] = {}
    for s, c in zip(sals.tolist(), codes.tolist()):
        sums[c] = sums.get(c, 0.0) + s
        counts[c] = counts.get(c, 0) + 1
    unique = sorted(sums.keys())
    return unique, np.array([sums[c] / counts[c] for c in unique])


# (b) Векторизовано — boolean masking з int-порівнянням;
#     внутрішній цикл лише по ~10 унікальних кодах країн.
def vec_means(sals, codes):
    unique = np.unique(codes)
    return list(unique), np.array([sals[codes == c].mean() for c in unique])


idx_a, means_a = loop_means(sal_top, country_codes)
idx_b, means_b = vec_means(sal_top, country_codes)
assert idx_a == idx_b and np.allclose(means_a, means_b), "Розбіжність!"

# Покажемо результат відсортовано за середньою зарплатою
order = np.argsort(means_b)[::-1]
print(f"{'Країна':<35} {'Середня зарплата ($)':>22}")
print("-" * 58)
for i in order:
    print(f"{country_names[idx_b[i]]:<35} {means_b[i]:>22,.0f}")


In [ ]:
# Швидкість — наївний цикл
%timeit loop_means(sal_top, country_codes)


In [ ]:
# Швидкість — векторизовано
%timeit vec_means(sal_top, country_codes)


**Чому векторизована версія швидша:** наївний цикл робить десятки тисяч Python-ітерацій з upsert у словник на кожному кроці. Векторизована — лише ~10 ітерацій по унікальних кодах країн, а вся важка робота (`mask = codes == c` + `.mean()`) відбувається у C-коді NumPy на суцільних масивах int-ів.

> **Чому ми кодували рядки в int.** На object-dtype масиві (наш `ctr_top`) операція `ctr_top == c` змушена робити **Python-порівняння для кожного рядка** — без жодного SIMD. Один раз перетворили назви на int-коди — і той самий `==` ганяється у C на суцільному `int64`-масиві у сотні разів швидше. Це універсальний прийом для будь-якої агрегації по строкових категоріях.

> **Note:** `vec_means` усе ж має зовнішній Python-цикл по унікальних кодах — але їх лише ~10, а не ~50 000 рядків. Це і є типовий патерн: **кладемо в Python-цикл лише те, що неминуче**, а решту — у NumPy.


## Топ-K і частоти: `np.argsort`, `np.unique(return_counts=True)`

Дві операції, які ви робитимете щодня в роботі з даними:

- **Топ-K елементів** — `np.argsort(arr)` повертає **індекси, які впорядкували б масив**. Беремо останні K — це топ-K за зростанням; розгортаємо `[::-1]` — за спаданням.
- **Частоти / value counts** — `np.unique(arr, return_counts=True)` повертає (унікальні значення, скільки разів кожне зустрічається).


In [ ]:
# Топ-10 найвищих зарплат у датасеті
k = 10
top_idx = np.argsort(salaries)[-k:][::-1]   # індекси топ-10, від найбільшої
top_salaries = salaries[top_idx]
top_countries_for_top = countries[top_idx]

print(f"Топ-{k} найвищих зарплат у Survey 2025:")
for s, c in zip(top_salaries, top_countries_for_top):
    print(f"  ${s:>15,.0f}   {c}")


In [ ]:
# Частоти країн — value counts на чистому NumPy
unique_countries, counts = np.unique(countries, return_counts=True)

# Сортуємо за counts ↓ і беремо топ-10
order = np.argsort(counts)[::-1][:10]
print(f"{'Країна':<35} {'Респондентів':>15}")
print("-" * 51)
for i in order:
    print(f"{unique_countries[i]:<35} {counts[i]:>15,}")

print(f"\nВсього країн у вибірці: {len(unique_countries)}")
print(f"Респондентів сумарно:    {counts.sum():,}")


## Функції втрат — мінімально (MSE, MAE, RMSE)

![Curve-Fitting — xkcd 2048](https://imgs.xkcd.com/comics/curve_fitting.png)

*[xkcd 2048 — Curve-Fitting](https://xkcd.com/2048/) · Randall Munroe · [CC BY-NC 2.5](https://xkcd.com/license.html)*

**Функція втрат** — це число, яке вимірює, наскільки прогноз відрізняється від істини. Сьогодні ми **не тренуємо** нічого — ми лише дивимось, як ці формули елегантно записуються одним рядком NumPy.

| Назва | Формула | NumPy | Особливість |
|-------|---------|-------|-------------|
| **MSE** (Mean Squared Error) | $\frac{1}{n}\sum (y - \hat{y})^2$ | `((y - yh) ** 2).mean()` | Карає великі помилки сильніше |
| **MAE** (Mean Absolute Error) | $\frac{1}{n}\sum \lvert y - \hat{y} \rvert$ | `np.abs(y - yh).mean()` | Лінійна; стійка до викидів |
| **RMSE** (Root MSE) | $\sqrt{\text{MSE}}$ | `np.sqrt(mse)` | Та сама розмірність, що й `y` |

Зверніть увагу: **жодного циклу**, жодного `for`. Уся арифметика відбувається у C-коді NumPy на суцільних масивах.


In [ ]:
# Синтетичні дані: "істина" та "прогноз" з невеликим шумом
rng = np.random.default_rng(SEED)
n = 1_000_000
y_true = rng.standard_normal(n)
y_pred = y_true + rng.standard_normal(n) * 0.5   # шум зі стандартним відхиленням 0.5

# Усі три функції втрат — один рядок кожна
mse = ((y_true - y_pred) ** 2).mean()
mae = np.abs(y_true - y_pred).mean()
rmse = np.sqrt(mse)

print(f"MSE  = {mse:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}   ← у тих самих одиницях, що і y")


### Швидкість: NumPy MSE vs Python-цикл

Перевіримо, скільки коштує "ручний" MSE на Python-списках, і порівняємо з NumPy-однорядковиком.

In [ ]:
y_true_list = y_true.tolist()
y_pred_list = y_pred.tolist()


def mse_python(y, yh):
    total = 0.0
    for a, b in zip(y, yh):
        total += (a - b) ** 2
    return total / len(y)


# Перевіримо, що результати збігаються
assert np.isclose(mse_python(y_true_list, y_pred_list), mse)


In [ ]:
%timeit mse_python(y_true_list, y_pred_list)


In [ ]:
%timeit ((y_true - y_pred) ** 2).mean()


Очікуйте різницю у **сотні разів**. Та сама математика — але один рядок NumPy виконується SIMD-інструкціями процесора на суцільному блоці пам'яті, тоді як Python-цикл робить мільйон диспатчів інтерпретатора.

## Pairwise distances через broadcasting

Класична задача: дано `n` точок у `d`-вимірному просторі — побудувати матрицю `(n, n)`, де елемент `[i, j]` = евклідова відстань між точками `i` та `j`.

**Наївне рішення** — два вкладені цикли. Складність `O(n²·d)`, але кожна ітерація дорого коштує через Python.

**Векторизоване** через broadcasting — буквально один вираз. Ключовий трюк:

- `points[:, None, :]` має форму `(n, 1, d)` — "стовпчик" точок.
- `points[None, :, :]` має форму `(1, n, d)` — "рядок" точок.
- Різниця через broadcasting дає `(n, n, d)` — для кожної пари (i, j) повний вектор-різниця.
- Підносимо до квадрата, сумуємо по останній осі (`axis=-1`), беремо корінь — і маємо матрицю `(n, n)` відстаней.

Це той самий патерн, який лежить в основі kNN, k-means, attention-механізмів у Transformer-ах та десятків інших алгоритмів.


In [ ]:
# 200 точок у 3D
rng_pts = np.random.default_rng(SEED)
n_pts, dim = 200, 3
points = rng_pts.standard_normal((n_pts, dim))


def pairwise_loop(pts):
    n = len(pts)
    out = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            diff = pts[i] - pts[j]
            out[i, j] = np.sqrt((diff * diff).sum())
    return out


def pairwise_vec(pts):
    # (n, 1, d) - (1, n, d) -> (n, n, d) ; sum по d ; sqrt
    diff = pts[:, None, :] - pts[None, :, :]
    return np.sqrt((diff * diff).sum(axis=-1))


D_loop = pairwise_loop(points)
D_vec = pairwise_vec(points)

assert np.allclose(D_loop, D_vec)
print(f"Матриця відстаней: shape = {D_vec.shape}")
print(f"Діагональ (відстань точки до самої себе) має бути 0: max = {np.diag(D_vec).max():.2e}")
print(f"Найдальша пара точок:  {D_vec.max():.3f}")
print(f"Середня попарна відстань: {D_vec[np.triu_indices(n_pts, k=1)].mean():.3f}")


In [ ]:
%timeit pairwise_loop(points)


In [ ]:
%timeit pairwise_vec(points)


Очікуйте прискорення у **сотні разів** на 200 точках — і воно лише зростає з `n`. На `n = 1000` цикл вже непрактичний, а вектор працює за мілісекунди.

> **Пам'ять:** проміжний тензор `diff` має форму `(n, n, d)` — для `n = 10 000` це вже 800 МБ при `float64`. Для великих `n` використовують або `scipy.spatial.distance.cdist`, або алгебричну тотожність `‖a − b‖² = ‖a‖² + ‖b‖² − 2·a·b` з GEMM-множенням — без проміжного `(n, n, d)`-тензора.

## Підсумок

**Чому NumPy швидкий.** Суцільна пам'ять + фіксований dtype + диспатч до C/SIMD = 10×–500× прискорення проти Python-циклів. Розмір масиву перетворює "помітно швидше" на "інший порядок величини".

**Що ми робили практично.**

- **Описова статистика** одним рядком: `arr.mean()`, `np.median(arr)`, `np.percentile(arr, [25, 50, 75, 99])`.
- **Виявлення викидів через IQR** — `(arr < lower) | (arr > upper)` дає boolean-маску; `mask.sum()` рахує, `arr[~mask]` фільтрує.
- **Агрегація по групах** — boolean masking + `.mean()`. Зовнішній цикл лише по унікальних групах, не по рядках.
- **Топ-K** — `np.argsort(arr)[-k:][::-1]`. **Частоти** — `np.unique(arr, return_counts=True)`.
- **Функції втрат** (MSE, MAE, RMSE) — формула один-в-один перекладається в один рядок NumPy.
- **Pairwise distances** — `points[:, None, :] - points[None, :, :]` + `sum(axis=-1)` + `sqrt`. Та сама ідея лежить в основі kNN, k-means і attention.

**Інженерні правила, які варто запам'ятати.**

- Basic slicing → **view** (без копіювання); fancy/boolean indexing → **copy**.
- Broadcasting не копіює дані у пам'яті — це "віртуальне" розтягування.
- Завжди питайте себе: "скільки тут Python-ітерацій?". Якщо відповідь — "по кожному елементу" — це привід векторизувати.

**Що ми навмисно не робили сьогодні.** Ми не тренували жодну модель, не торкалися градієнтного спуску, метрик класифікації чи `scikit-learn`. Це теми **наступного року** — і ваш сьогоднішній NumPy-фундамент зробить їх у рази простішими.


## Джерела

**Офіційна документація:**

- [NumPy: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) — канонічний опис правил broadcasting з ілюстраціями.
- [NumPy: Indexing on `ndarray`s](https://numpy.org/doc/stable/user/basics.indexing.html) — basic vs fancy vs boolean.
- [NumPy: `numpy.percentile`](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html), [`numpy.unique`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html), [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) — функції, які ми використовували сьогодні.
- [NumPy: Statistics routines](https://numpy.org/doc/stable/reference/routines.statistics.html) — повний перелік статистичних функцій.

**Книги:**

- Wes McKinney — *Python for Data Analysis* (3rd ed., O'Reilly, 2022). Розділ 4 "NumPy Basics" і Розділ 12 "Advanced NumPy" — найкращий друкований ресурс.

**Дотичне читання:**

- John W. Tukey — *Exploratory Data Analysis* (1977). Першоджерело правила `1.5 · IQR` для викидів.
- [SciPy: `scipy.spatial.distance.cdist`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html) — для великих `n` у задачі pairwise distances.

**Датасет:**

- [Stack Overflow Annual Developer Survey 2025](https://survey.stackoverflow.co/2025/) — Open Database License (ODbL).


## Що далі?

**Лекція 13 — Візуалізація (matplotlib + seaborn).** Усі сьогоднішні `np.histogram`-подібні розподіли зарплат, IQR-фільтри, топ-K та матриці відстаней нарешті стануть **картинками**. Той самий NumPy-масив, який ми навчилися обчислювати — але вже з осями, легендами й кольорами.

**Куди ще веде NumPy** (поза курсом цього семестру):

- `scipy` — оптимізація, статистика, лінійна алгебра, обробка сигналів — усе на NumPy-масивах.
- `pandas` — те, що ми бачили в Лекції 11, всередині використовує NumPy для всіх числових операцій.
- ML-фреймворки — `scikit-learn`, `pytorch`, `jax` — приймають NumPy на вході, а їхні внутрішні тензори побудовані за тими самими принципами (суцільна пам'ять + фіксований dtype + векторизовані операції).

Все, що ви сьогодні навчилися, ви впізнáєте у будь-якому з цих інструментів.
